# llama.cpp Server — Test Notebook

Tests the running `llamacpp` container (port `10000`) via its OpenAI-compatible API.

Make sure the server is up:
```bash
docker compose up -d
docker compose logs -f llamacpp
```

In [30]:
%pip install -q openai

/home/alif-pc/Codes/inference-engine-deployment/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [31]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:10000/v1", api_key="none")

## Text test

In [32]:
resp = client.chat.completions.create(
    model="qwen",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Hello.",
        }
    ],
    max_tokens=32,
    temperature=0.7,
    top_p=0.8,
    extra_body={
        "chat_template_kwargs": {"enable_thinking": False},
    },
)

msg = resp.choices[0].message
content = (msg.content or "").strip()
reasoning = (getattr(msg, "reasoning_content", "") or "").strip()

print("=== Response ===")
print(content if content else "[EMPTY_CONTENT]")
print(f"content_len={len(content)} reasoning_len={len(reasoning)}")

=== Response ===
Hello.
content_len=6 reasoning_len=0


In [33]:
import time
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:10000/v1",
    timeout=3600,
)

messages = [
    {
        "role": "system",
        "content": "Answer briefly in one sentence.",
    },
    {
        "role": "user",
        "content": "Why did image input fail earlier?",
    },
]

start = time.time()
response = client.chat.completions.create(
    model="Qwen3.5-27B-Q3_K_M.gguf",
    messages=messages,
    max_tokens=64,
    temperature=0.7,
    top_p=0.8,
    extra_body={
        "chat_template_kwargs": {"enable_thinking": False},
    },
)

msg = response.choices[0].message
content = (msg.content or "").strip()
reasoning = (getattr(msg, "reasoning_content", "") or "").strip()
print(f"Response costs: {time.time() - start:.2f}s")
print(f"Generated text: {content if content else '[EMPTY_CONTENT]'}")
print(f"content_len={len(content)} reasoning_len={len(reasoning)}")

Response costs: 0.95s
Generated text: I cannot determine the specific cause of your earlier image input failure as I do not have access to your session history or the error details you encountered.
content_len=159 reasoning_len=0


## LangChain + LangGraph quick test

This section uses your local OpenAI-compatible endpoint at `http://localhost:10000/v1` to evaluate concise responses and latency.

In [34]:
import time
from langchain_openai import ChatOpenAI

lc_model = ChatOpenAI(
    api_key="EMPTY",
    base_url="http://localhost:10000/v1",
    model="Qwen3.5-27B-Q3_K_M.gguf",
    temperature=0.7,
    max_tokens=64,
    model_kwargs={
        "top_p": 0.8,
        "extra_body": {
            "chat_template_kwargs": {"enable_thinking": False},
        },
    },
)

start = time.time()
lc_resp = lc_model.invoke("Reply in <= 8 words: say hello.")
print(f"LangChain latency: {time.time() - start:.2f}s")
print("LangChain output:", (lc_resp.content or "").strip() or "[EMPTY_CONTENT]")

LangChain latency: 0.44s
LangChain output: Hello! How can I help you today?


In [35]:
import time
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class FlowState(TypedDict):
    question: str
    answer: str


def call_model(state: FlowState):
    t0 = time.time()
    out = lc_model.invoke(state["question"]).content or ""
    print(f"LangGraph node latency: {time.time() - t0:.2f}s")
    return {"answer": out.strip()}


graph_builder = StateGraph(FlowState)
graph_builder.add_node("call_model", call_model)
graph_builder.add_edge(START, "call_model")
graph_builder.add_edge("call_model", END)
graph = graph_builder.compile()

result = graph.invoke({"question": "Reply in <= 8 words: what failed earlier?"})
print("LangGraph output:", result["answer"] or "[EMPTY_CONTENT]")

LangGraph node latency: 0.44s
LangGraph output: I do not have context on previous failures.


## Mini benchmark matrix (OpenAI vs LangChain vs LangGraph)

Runs each client multiple times with the same prompt and reports:
- average latency (s)
- p95 latency (s)
- empty content rate
- average content length

In [38]:
import time
from typing import TypedDict

import numpy as np
import pandas as pd
from IPython.display import display
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

BASE_URL = "http://localhost:10000/v1"
MODEL = "Qwen3.5-27B-Q3_K_M.gguf"
N_RUNS = 5
PROMPT = "Reply in <= 8 words: say hello."


def p95(values):
    if not values:
        return 0.0
    return float(np.percentile(values, 95))


client = OpenAI(api_key="EMPTY", base_url=BASE_URL, timeout=3600)


def run_openai_once():
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=64,
        temperature=0.7,
        top_p=0.8,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    dt = time.time() - t0
    msg = resp.choices[0].message
    content = (msg.content or "").strip()
    return dt, content


lc_model = ChatOpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    model=MODEL,
    temperature=0.7,
    max_tokens=64,
    model_kwargs={
        "top_p": 0.8,
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    },
)


def run_langchain_once():
    t0 = time.time()
    out = lc_model.invoke(PROMPT)
    dt = time.time() - t0
    content = (out.content or "").strip()
    return dt, content


class BenchState(TypedDict):
    question: str
    answer: str


def _node(state: BenchState):
    out = lc_model.invoke(state["question"]).content or ""
    return {"answer": out.strip()}


graph_builder = StateGraph(BenchState)
graph_builder.add_node("call_model", _node)
graph_builder.add_edge(START, "call_model")
graph_builder.add_edge("call_model", END)
bench_graph = graph_builder.compile()


def run_langgraph_once():
    t0 = time.time()
    result = bench_graph.invoke({"question": PROMPT})
    dt = time.time() - t0
    content = (result.get("answer") or "").strip()
    return dt, content


def run_suite(name, fn, n_runs=N_RUNS):
    lat = []
    contents = []
    errors = 0
    for _ in range(n_runs):
        try:
            dt, content = fn()
            lat.append(dt)
            contents.append(content)
        except Exception:
            errors += 1

    non_empty = [c for c in contents if c]
    empty_rate = (len([c for c in contents if not c]) / len(contents)) if contents else 1.0

    return {
        "Client": name,
        "Runs": len(contents),
        "Errors": errors,
        "Avg Latency (s)": (sum(lat) / len(lat)) if lat else 0.0,
        "P95 Latency (s)": p95(lat) if lat else 0.0,
        "Empty Rate": empty_rate,
        "Avg Content Len": (sum(len(c) for c in non_empty) / len(non_empty)) if non_empty else 0.0,
    }


rows = [
    run_suite("openai", run_openai_once),
    run_suite("langchain", run_langchain_once),
    run_suite("langgraph", run_langgraph_once),
]

results_df = pd.DataFrame(rows).sort_values("Client").reset_index(drop=True)

styled = (
    results_df.style
    .format({
        "Avg Latency (s)": "{:.3f}",
        "P95 Latency (s)": "{:.3f}",
        "Empty Rate": "{:.0%}",
        "Avg Content Len": "{:.1f}",
    })
)

print(f"Benchmark runs per client: {N_RUNS}")
display(styled)

/home/alif-pc/Codes/inference-engine-deployment/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3641: UserWarning: Parameters {'extra_body', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


Benchmark runs per client: 5


,Client,Runs,Errors,Avg Latency (s),P95 Latency (s),Empty Rate,Avg Content Len
0,langchain,5,0,0.327,0.339,0%,30.0
1,langgraph,5,0,0.327,0.339,0%,29.6
2,openai,5,0,0.361,0.456,0%,30.8


## Multi-user simulation (different chats / subagent personas)

This test simulates concurrent users where each user has:
- its own chat history (multi-turn)
- an optional persona/system prompt (subagent-style behavior)

Tune `CONCURRENCY` and `USERS` to stress the server.

In [ ]:
import asyncio
import random
import time
import numpy as np
import pandas as pd
from openai import AsyncOpenAI
from IPython.display import display

BASE_URL = "http://localhost:10000/v1"
MODEL = "Qwen3.5-27B-Q3_K_M.gguf"

# Scale these up for heavier tests
USERS = 12
CONCURRENCY = 4
TURNS_PER_USER = 2
MAX_TOKENS = 64

# Optional persona prompts (subagent-style behavior)
PERSONAS = [
    "You are concise and factual.",
    "You are friendly and brief.",
    "You are a debugging assistant.",
    "You summarize in one short sentence.",
]

USER_PROMPTS = [
    "Give a short hello.",
    "Explain in one sentence why image input failed earlier.",
    "Summarize the benchmark result style you saw.",
    "Give one tip to reduce empty responses.",
]

client_async = AsyncOpenAI(api_key="EMPTY", base_url=BASE_URL, timeout=3600)


async def run_user_chat(user_id: int, sem: asyncio.Semaphore):
    """Simulate one user with its own conversation history."""
    latencies = []
    empty_count = 0
    contents = []

    persona = random.choice(PERSONAS)
    history = [{"role": "system", "content": persona}]

    for turn in range(TURNS_PER_USER):
        prompt = random.choice(USER_PROMPTS)
        history.append({"role": "user", "content": prompt})

        async with sem:
            t0 = time.time()
            resp = await client_async.chat.completions.create(
                model=MODEL,
                messages=history,
                max_tokens=MAX_TOKENS,
                temperature=0.7,
                top_p=0.8,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            )
            dt = time.time() - t0

        msg = resp.choices[0].message
        content = (msg.content or "").strip()

        latencies.append(dt)
        contents.append(content)
        if not content:
            empty_count += 1

        history.append({"role": "assistant", "content": content})

    return {
        "user_id": user_id,
        "turns": TURNS_PER_USER,
        "avg_latency_s": sum(latencies) / len(latencies),
        "p95_latency_s": float(np.percentile(latencies, 95)) if latencies else 0.0,
        "empty_rate": empty_count / TURNS_PER_USER,
        "avg_content_len": (sum(len(c) for c in contents if c) / max(1, len([c for c in contents if c]))),
    }


async def run_multi_user_test():
    sem = asyncio.Semaphore(CONCURRENCY)
    tasks = [run_user_chat(i, sem) for i in range(USERS)]
    rows = await asyncio.gather(*tasks)
    return rows

rows = await run_multi_user_test()
df = pd.DataFrame(rows).sort_values("user_id").reset_index(drop=True)

summary = {
    "users": USERS,
    "concurrency": CONCURRENCY,
    "turns_per_user": TURNS_PER_USER,
    "overall_avg_latency_s": round(df["avg_latency_s"].mean(), 3),
    "overall_p95_latency_s": round(float(np.percentile(df["p95_latency_s"].to_numpy(), 95)), 3),
    "overall_empty_rate": round(df["empty_rate"].mean(), 3),
    "overall_avg_content_len": round(df["avg_content_len"].mean(), 1),
}

print("=== Multi-user summary ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\n=== Per-user table ===")
display(
    df.style.format(
        {
            "avg_latency_s": "{:.3f}",
            "p95_latency_s": "{:.3f}",
            "empty_rate": "{:.0%}",
            "avg_content_len": "{:.1f}",
        }
    )
)

=== Multi-user summary ===
users: 12
concurrency: 4
turns_per_user: 2
overall_avg_latency_s: 2.635
overall_p95_latency_s: 6.577
overall_empty_rate: 0.0
overall_avg_content_len: 140.8

=== Per-user table ===


,user_id,turns,avg_latency_s,p95_latency_s,empty_rate,avg_content_len
0,0,2,6.059,6.918,0%,273.5
1,1,2,3.924,6.298,0%,176.0
2,2,2,3.501,3.811,0%,79.5
3,3,2,3.031,4.705,0%,97.0
4,4,2,4.290,4.346,0%,348.5
5,5,2,3.639,5.098,0%,272.0
6,6,2,1.849,1.979,0%,125.5
7,7,2,0.744,1.093,0%,38.5
8,8,2,0.984,1.525,0%,52.5
9,9,2,1.526,2.654,0%,113.0
